# Experiment 01: Zero-Leakage Data Splitting & Preprocessing Inspection

This notebook verifies:
1. **TIS-620 Character Decoding**: Decimal byte value to Thai glyph character mapping.
2. **Deterministic Group-Aware 80/20 Splitting**: Partitioning without data leakage.
3. **Zero Data Leakage Verification**: Proof that document pairs (`_sg`/`_tg`) and pages are strictly isolated.
4. **Letterbox Square Padding**: Aspect-ratio preserving padding to $32\times 32$ / $64\times 64$.
5. **Domain-Specific Thai Glyph Augmentations**: Morphological dilation/erosion, bounded rotation.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import torch

# Add src directory to path
src_dir = (Path.cwd() / '..' / 'src').resolve()
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dataset import build_split_dataframes, letterbox_pad, tis620_to_char
from transforms import get_train_transform, MorphologicalTransform

DATASET_DIR = (Path.cwd() / '..' / '..' / '..' / 'ThaiCharacter Dataset' / 'round2').resolve()
print(f'Dataset Directory: {DATASET_DIR}')
print(f'CUDA Available: {torch.cuda.is_available()}')


## 1. Run Deterministic Group-Aware 80/20 Split

In [ ]:
train_df, test_df, class_to_idx = build_split_dataframes(DATASET_DIR, train_ratio=0.8)

total_samples = len(train_df) + len(test_df)
print(f'Total Images: {total_samples:,}')
print(f'Train Samples: {len(train_df):,} ({len(train_df)/total_samples*100:.2f}%)')
print(f'Test Samples:  {len(test_df):,} ({len(test_df)/total_samples*100:.2f}%)')
print(f'Unique Classes in Train: {train_df["class_idx"].nunique()} / {len(class_to_idx)}')
print(f'Unique Classes in Test:  {test_df["class_idx"].nunique()} / {len(class_to_idx)}')


## 2. Zero-Leakage Verification
Ensure that no document source/page grouping appears in both train and test splits.

In [ ]:
train_groups = set(train_df['group_key'].unique())
test_groups = set(test_df['group_key'].unique())
leakage = train_groups.intersection(test_groups)

print(f'Unique Train Document-Page Groups: {len(train_groups)}')
print(f'Unique Test Document-Page Groups:  {len(test_groups)}')
print(f'Overlapping Groups (Data Leakage): {len(leakage)}')
assert len(leakage) == 0, 'Data leakage detected between train and test groups!'
print('✅ PASS: Zero Data Leakage verified!')


## 3. Visualize Aspect-Preserving Letterbox Padding & Morphological Augmentations

In [ ]:
sample_rows = train_df.sample(8, random_state=42)
morph = MorphologicalTransform(p_dilate=0.5, p_erode=0.5, kernel_size=2)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for col_idx, (_, row) in enumerate(sample_rows.iterrows()):
    with Image.open(row['filepath']) as raw_img:
        padded = letterbox_pad(raw_img.convert('RGB'), target_size=(32, 32))
        augmented = morph(padded)
        
        axes[0, col_idx].imshow(padded)
        axes[0, col_idx].set_title(f"{row['class_number']}: {row['character']}\n(Letterboxed)", fontsize=10)
        axes[0, col_idx].axis('off')
        
        axes[1, col_idx].imshow(augmented)
        axes[1, col_idx].set_title(f"{row['character']}\n(Morph Aug)", fontsize=10)
        axes[1, col_idx].axis('off')

plt.suptitle('Original Letterboxed (32x32) vs Morphological Stroke Augmentation', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
